In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

from google import genai
from google.genai import types

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

transcript = client.models.generate_content(
   model="gemini-3.5-flash",
   contents="""Generate a short transcript around 100 words that reads
            like it was clipped from a podcast by excited herpetologists.
            The hosts names are Dr. Anya and Liam.""").text

response = client.models.generate_content(
   model="gemini-3.1-flash-tts-preview",
   contents=transcript,
   config=types.GenerateContentConfig(
      response_modalities=["AUDIO"],
      speech_config=types.SpeechConfig(
         multi_speaker_voice_config=types.MultiSpeakerVoiceConfig(
            speaker_voice_configs=[
               types.SpeakerVoiceConfig(
                  speaker='Dr. Anya',
                  voice_config=types.VoiceConfig(
                     prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Kore',
                     )
                  )
               ),
               types.SpeakerVoiceConfig(
                  speaker='Liam',
                  voice_config=types.VoiceConfig(
                     prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Puck',
                     )
                  )
               ),
            ]
         )
      )
   )
)

# ...Code to handle audio output

In [2]:
from IPython.display import Audio

# 1. Grab the audio bytes from the response parts
audio_bytes = response.candidates[0].content.parts[0].inline_data.data

# 2. Display the interactive audio player
Audio(audio_bytes, rate=24000)  # TTS preview usually outputs at 24kHz

In [3]:
# 1. Grab the audio bytes
audio_bytes = response.candidates[0].content.parts[0].inline_data.data

# 2. Write the bytes to a file
output_filename = "podcast_clip.mp3"
with open(output_filename, "wb") as f:
    f.write(audio_bytes)

print(f"Audio successfully saved to {output_filename}")

Audio successfully saved to podcast_clip.mp3


In [4]:
from IPython.display import Audio

# 1. Grab the audio bytes
audio_bytes = response.candidates[0].content.parts[0].inline_data.data

# 2. Save it locally first
with open("temp_podcast.wav", "wb") as f:
    f.write(audio_bytes)

# 3. Feed the file path to the player instead of raw bytes
Audio("temp_podcast.wav")

In [6]:
import wave

# 1. Safely extract the raw audio bytes from your response object
audio_bytes = response.candidates[0].content.parts[0].inline_data.data

# 2. Define the output file name
output_filename = "podcast_output.wav"

# 3. Write it with explicit audio properties (1 channel, 2 bytes/sample, 24kHz)
with wave.open(output_filename, "wb") as wav_file:
    wav_file.setnchannels(1)       # Mono audio
    wav_file.setsampwidth(2)       # 16-bit audio (2 bytes per sample)
    wav_file.setframerate(24000)   # Gemini TTS uses a 24kHz sample rate
    wav_file.writeframes(audio_bytes)

print(f"Success! Audio converted and saved as: {output_filename}")

Success! Audio converted and saved as: podcast_output.wav


In [7]:
transcript

'**[Audio Clip Begins]**\n\n**Liam:** —and that’s when Anya literally dove headfirst into the mud!\n\n**Dr. Anya:** *(Laughs)* Liam, I had to! It was a *Boophis albilabris*! \n\n**Liam:** Guys, you have no idea. Under the UV light, this tiny tree frog was glowing like a neon highway sign! A brilliant, bio-fluorescent turquoise. \n\n**Dr. Anya:** We’re talking about a species undocumented in this valley for fifty years, just sitting on a leaf, shining like a little emerald! \n\n**Liam:** My heart was hammering. I think I forgot to breathe for a solid minute. \n\n**Dr. Anya:** We got the DNA swabs, the photos, and—oh, just wait till you see the video! \n\n**[Audio Clip Ends]**'

In [3]:
import os
import wave
from dotenv import load_dotenv
from google import genai
from google.genai import types

# 1. Initialize Environment & Client
load_dotenv()
client = genai.Client()

# 2. Your Exact Isolated Ad Script Text
ad_script_text = """
Most online store programs work like this: they get paid when you buy their course. Mine is different. I only get paid when *your* store actually moves inventory.

Look, I understand if you're skeptical. You've been right to be before. Maybe you’ve tried one of these online store things and it just felt like a casino. You saw the red flags, hoped it would be different, and now feel a little stupid for trusting your judgment. It cost you a few thousand dollars and maybe some pride. Your spouse might even be wondering if you'll follow through. You’re asking, "What’s the catch *this* time?"

The truth is, it wasn't your judgment that was wrong. It was the model they were selling you. What if there's a different way to build a real online storefront, selling products already in demand? A way to earn supplemental income without the upfront gamble or the stress of finding products. A way to build quiet confidence, and a little more breathing room. An earned calm.

I’m Marcus Vane. I ran wholesale logistics for 14 years before starting the Sovereign Storefront program 5 years ago. I saw warehouses packed with perfectly good returned inventory, and regular folks trying to sell online without knowing what to get, or where.

My system helps you build an online consignment storefront, reselling that inventory from our vetted regional liquidator partners. We provide a weekly 'Liquidator Match Sheet' – inventory with existing demand, ready to be cataloged. The difference? I only get paid when *your* store moves their products. Liquidators pay *me* a placement fee, so my income is tied directly to your success. It’s a completely different incentive.

As member Dana R. said, "I was sure this was another version of the thing that cost me $2,400. Then he explained who actually pays him and it clicked. I have a real store now, not a folder of unwatched videos." With over 9,400 members globally, and these match sheets running for five years, this is a real operational system, not a get-rich-quick scheme.

If you're ready to see how this operational system actually works, without the hype or empty promises, watch our First Pallet workshop. It's a two-hour deep dive into the mechanism, including a sample match sheet and store starter kit. It costs just $1. The link is below.
"""

# 3. Post-Process the Script via Gemini for Pacing & Audio Directing
director_prompt = f"""
You are an expert direct-response audio director. Take this specific ad copy text and rewrite it into an optimized Speech Synthesis format. 
Keep every single word identical, but insert explicit timing flags using `<break time="XXXms"/>` tags.

Follow these pacing directions for the character (Marcus Vane - an unhurried, steady, conversational logistics veteran):
- Insert an explicit 1000ms emotional break after heavy sentences involving spousal doubt, losing money, or feeling stupid.
- Insert a 600ms pacing break between major thematic paragraphs (e.g., after "An earned calm.").
- Insert natural 300ms breathing room around emphasis transitions (e.g., after "Mine is different.").
- Return ONLY the clean, wrapped, marked-up text inside a single <speak> block. Do not include any explanation.

Script to process:
{ad_script_text}
"""

print("--- Step 1: Generating Audio Director Pacing Tags ---")
director_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=director_prompt
)
marked_up_script = director_response.text
# Strip out any markdown wrappers if the director model adds them
marked_up_script = director_response.text.replace("```xml", "").replace("```html", "").replace("```", "").strip()
print(marked_up_script)

# 4. Generate the Final Audio Track using the High-Fidelity Enceladus Voice
print("\n--- Step 2: Synthesizing Voiceover with en-US-Chirp3-HD-Enceladus ---")
voice_response = client.models.generate_content(
   model="gemini-3.1-flash-tts-preview", 
   contents=marked_up_script,
   config=types.GenerateContentConfig(
      response_modalities=["AUDIO"],
      speech_config=types.SpeechConfig(
          voice_config=types.VoiceConfig(
              prebuilt_voice_config=types.PrebuiltVoiceConfig(
                  voice_name='Enceladus'
              )
          )
      )
   )
)

# ==========================================
# 5. Save the Output safely by inspecting parts
# ==========================================
import base64

generated_part = voice_response.candidates[0].content.parts[0]
audio_bytes = None

# Look inside the native part object structure
if hasattr(generated_part, 'inline_data') and generated_part.inline_data:
    raw_data = generated_part.inline_data.data
    
    # If the SDK returns it as a bytes object already, use it directly
    if isinstance(raw_data, bytes):
        audio_bytes = raw_data
    # If it's returned as a base64 string, decode it to raw PCM/WAV bytes
    elif isinstance(raw_data, str):
        audio_bytes = base64.b64decode(raw_data)

# Fallback: check if the SDK response container has a top-level reading helper
if not audio_bytes and hasattr(voice_response, 'read_bytes'):
    try:
        audio_bytes = voice_response.read_bytes()
    except Exception:
        pass

if not audio_bytes:
    # Debug print so you can see exactly what the model returned if it fails again
    print("\n[DEBUG] Part Structure:", type(generated_part), dir(generated_part))
    if hasattr(generated_part, 'inline_data'):
        print("[DEBUG] inline_data type:", type(generated_part.inline_data))
    raise ValueError("Could not find raw audio bytes in the response object. Check voice_response structure.")

output_filename = "marcus_final_script.wav"

with wave.open(output_filename, "wb") as wav_file:
    wav_file.setnchannels(1)       # Mono
    wav_file.setsampwidth(2)       # 16-bit PCM
    wav_file.setframerate(24000)   # 24kHz Sample rate for high-fidelity Chirp3
    wav_file.writeframes(audio_bytes)

print(f"\nSuccess! Ad voiceover written cleanly to: {output_filename}")

--- Step 1: Generating Audio Director Pacing Tags ---
<speak>Most online store programs work like this: they get paid when you buy their course. Mine is different.<break time="300ms"/> I only get paid when *your* store actually moves inventory.
Look, I understand if you're skeptical. You've been right to be before. Maybe you’ve tried one of these online store things and it just felt like a casino. You saw the red flags, hoped it would be different, and now feel a little stupid for trusting your judgment.<break time="1000ms"/> It cost you a few thousand dollars and maybe some pride.<break time="1000ms"/> Your spouse might even be wondering if you'll follow through.<break time="1000ms"/> You’re asking, "What’s the catch *this* time?"
The truth is, it wasn't your judgment that was wrong. It was the model they were selling you. What if there's a different way to build a real online storefront, selling products already in demand? A way to earn supplemental income without the upfront gamble 

In [4]:
import os
import wave
import base64
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()
client = genai.Client()

# Split your marked-up script into chunks by paragraph
def split_into_chunks(text, max_chars=800):
    """Split on double newlines, then merge small chunks up to max_chars."""
    paragraphs = [p.strip() for p in text.strip().split("\n\n") if p.strip()]
    chunks = []
    current = ""
    for para in paragraphs:
        if len(current) + len(para) < max_chars:
            current += ("\n\n" if current else "") + para
        else:
            if current:
                chunks.append(current)
            current = para
    if current:
        chunks.append(current)
    return chunks

# Run Step 1 (director markup) as before...
# marked_up_script = ... (your existing Step 1 code)

# Step 2: Chunked TTS synthesis
chunks = split_into_chunks(marked_up_script, max_chars=800)
print(f"\n--- Step 2: Synthesizing {len(chunks)} chunks ---")

all_audio_bytes = []

for i, chunk in enumerate(chunks):
    print(f"  Generating chunk {i+1}/{len(chunks)} ({len(chunk)} chars)...")
    
    voice_response = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=chunk,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Enceladus'
                    )
                )
            )
        )
    )
    
    generated_part = voice_response.candidates[0].content.parts[0]
    
    if hasattr(generated_part, 'inline_data') and generated_part.inline_data:
        raw_data = generated_part.inline_data.data
        if isinstance(raw_data, bytes):
            all_audio_bytes.append(raw_data)
        elif isinstance(raw_data, str):
            all_audio_bytes.append(base64.b64decode(raw_data))
    else:
        print(f"  [WARNING] Chunk {i+1} returned no audio — skipping.")

# Step 3: Concatenate all PCM chunks and write a single WAV
combined_pcm = b"".join(all_audio_bytes)
output_filename = "marcus_final_script.wav"

with wave.open(output_filename, "wb") as wav_file:
    wav_file.setnchannels(1)
    wav_file.setsampwidth(2)
    wav_file.setframerate(24000)
    wav_file.writeframes(combined_pcm)

print(f"\nSuccess! Combined {len(chunks)} chunks → {output_filename}")


--- Step 2: Synthesizing 1 chunks ---
  Generating chunk 1/1 (2456 chars)...

Success! Combined 1 chunks → marcus_final_script.wav


In [5]:
import os
import re
import wave
import base64
from dotenv import load_dotenv
from google import genai
from google.genai import types

# ============================================================
# 1. Initialize Environment & Client
# ============================================================
load_dotenv()
client = genai.Client()

# ============================================================
# 2. Ad Script Text
# ============================================================
ad_script_text = """
Most online store programs work like this: they get paid when you buy their course. Mine is different. I only get paid when *your* store actually moves inventory.

Look, I understand if you're skeptical. You've been right to be before. Maybe you've tried one of these online store things and it just felt like a casino. You saw the red flags, hoped it would be different, and now feel a little stupid for trusting your judgment. It cost you a few thousand dollars and maybe some pride. Your spouse might even be wondering if you'll follow through. You're asking, "What's the catch *this* time?"

The truth is, it wasn't your judgment that was wrong. It was the model they were selling you. What if there's a different way to build a real online storefront, selling products already in demand? A way to earn supplemental income without the upfront gamble or the stress of finding products. A way to build quiet confidence, and a little more breathing room. An earned calm.

I'm Marcus Vane. I ran wholesale logistics for 14 years before starting the Sovereign Storefront program 5 years ago. I saw warehouses packed with perfectly good returned inventory, and regular folks trying to sell online without knowing what to get, or where.

My system helps you build an online consignment storefront, reselling that inventory from our vetted regional liquidator partners. We provide a weekly 'Liquidator Match Sheet' – inventory with existing demand, ready to be cataloged. The difference? I only get paid when *your* store moves their products. Liquidators pay *me* a placement fee, so my income is tied directly to your success. It's a completely different incentive.

As member Dana R. said, "I was sure this was another version of the thing that cost me $2,400. Then he explained who actually pays him and it clicked. I have a real store now, not a folder of unwatched videos." With over 9,400 members globally, and these match sheets running for five years, this is a real operational system, not a get-rich-quick scheme.

If you're ready to see how this operational system actually works, without the hype or empty promises, watch our First Pallet workshop. It's a two-hour deep dive into the mechanism, including a sample match sheet and store starter kit. It costs just $1. The link is below.
"""

# ============================================================
# 3. Director Pass — Insert Pacing Tags via Gemini
# ============================================================
director_prompt = f"""
You are an expert direct-response audio director. Take this specific ad copy text and rewrite it into an optimized Speech Synthesis format. 
Keep every single word identical, but insert explicit timing flags using `<break time="XXXms"/>` tags.

Follow these pacing directions for the character (Marcus Vane - an unhurried, steady, conversational logistics veteran):
- Insert an explicit 1000ms emotional break after heavy sentences involving spousal doubt, losing money, or feeling stupid.
- Insert a 600ms pacing break between major thematic paragraphs (e.g., after "An earned calm.").
- Insert natural 300ms breathing room around emphasis transitions (e.g., after "Mine is different.").
- Return ONLY the clean, wrapped, marked-up text inside a single <speak> block. Do not include any explanation.

Script to process:
{ad_script_text}
"""

print("--- Step 1: Generating Audio Director Pacing Tags ---")
director_response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=director_prompt
)
marked_up_script = director_response.text
marked_up_script = marked_up_script.replace("```xml", "").replace("```html", "").replace("```", "").strip()

# Strip the outer <speak>...</speak> wrapper so we can re-wrap each chunk cleanly
inner_script = re.sub(r'^\s*<speak>\s*', '', marked_up_script)
inner_script = re.sub(r'\s*</speak>\s*$', '', inner_script).strip()

print(marked_up_script)


# ============================================================
# 4. Sentence-Aware Chunker (~800 chars per chunk)
# ============================================================
def split_into_sentence_chunks(text: str, max_chars: int = 800) -> list[str]:
    """
    Splits text into chunks of ~max_chars, always breaking on sentence
    boundaries (. ! ?) so no sentence is ever cut mid-way.
    Break tags are treated as zero-width for splitting purposes.
    """
    # Split on sentence-ending punctuation followed by whitespace
    sentence_pattern = re.compile(r'(?<=[.!?])\s+')
    sentences = sentence_pattern.split(text.strip())

    chunks = []
    current_chunk = ""

    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue

        candidate = (current_chunk + " " + sentence).strip() if current_chunk else sentence

        if len(candidate) <= max_chars:
            current_chunk = candidate
        else:
            # Current chunk is full — save it and start a new one
            if current_chunk:
                chunks.append(current_chunk.strip())
            # If a single sentence itself exceeds max_chars, it goes in alone
            current_chunk = sentence

    if current_chunk:
        chunks.append(current_chunk.strip())

    return chunks


chunks = split_into_sentence_chunks(inner_script, max_chars=800)
print(f"\n--- Step 2: Script split into {len(chunks)} chunks ---")
for i, chunk in enumerate(chunks):
    print(f"  Chunk {i+1}: {len(chunk)} chars — {chunk[:60]}...")


# ============================================================
# 5. TTS Synthesis — One API call per chunk
# ============================================================
def synthesize_chunk(text_chunk: str, chunk_index: int) -> bytes:
    """Wraps a chunk in <speak> tags and sends it to Gemini TTS."""
    ssml_chunk = f"<speak>{text_chunk}</speak>"

    response = client.models.generate_content(
        model="gemini-2.5-flash-preview-tts",
        contents=ssml_chunk,
        config=types.GenerateContentConfig(
            response_modalities=["AUDIO"],
            speech_config=types.SpeechConfig(
                voice_config=types.VoiceConfig(
                    prebuilt_voice_config=types.PrebuiltVoiceConfig(
                        voice_name='Enceladus'
                    )
                )
            )
        )
    )

    part = response.candidates[0].content.parts[0]

    if hasattr(part, 'inline_data') and part.inline_data:
        raw = part.inline_data.data
        if isinstance(raw, bytes):
            return raw
        elif isinstance(raw, str):
            return base64.b64decode(raw)

    raise ValueError(f"Chunk {chunk_index + 1}: No audio bytes returned. "
                     f"Part type: {type(part)}, attrs: {dir(part)}")


print("\n--- Step 3: Synthesizing chunks with Enceladus ---")
all_pcm_bytes = []

for i, chunk in enumerate(chunks):
    print(f"  Synthesizing chunk {i + 1}/{len(chunks)}...")
    pcm = synthesize_chunk(chunk, i)
    all_pcm_bytes.append(pcm)
    print(f"    ✓ {len(pcm):,} bytes received")


# ============================================================
# 6. Combine All PCM Chunks → Single WAV File
# ============================================================
combined_pcm = b"".join(all_pcm_bytes)
output_filename = "marcus_final_script.wav"

with wave.open(output_filename, "wb") as wav_file:
    wav_file.setnchannels(1)    # Mono
    wav_file.setsampwidth(2)    # 16-bit PCM
    wav_file.setframerate(24000)  # 24kHz — Chirp3-HD native rate
    wav_file.writeframes(combined_pcm)

total_seconds = len(combined_pcm) / (24000 * 2)
print(f"\n✅ Success! {len(chunks)} chunks merged → {output_filename}")
print(f"   Total audio duration: {int(total_seconds // 60)}m {int(total_seconds % 60)}s")
print(f"   File size: {os.path.getsize(output_filename):,} bytes")

--- Step 1: Generating Audio Director Pacing Tags ---
<speak>Most online store programs work like this: they get paid when you buy their course. Mine is different.<break time="300ms"/> I only get paid when *your* store actually moves inventory.

Look, I understand if you're skeptical. You've been right to be before. Maybe you've tried one of these online store things and it just felt like a casino. You saw the red flags, hoped it would be different, and now feel a little stupid for trusting your judgment.<break time="1000ms"/> It cost you a few thousand dollars and maybe some pride.<break time="1000ms"/> Your spouse might even be wondering if you'll follow through.<break time="1000ms"/> You're asking, "What's the catch *this* time?"<break time="600ms"/>

The truth is, it wasn't your judgment that was wrong. It was the model they were selling you. What if there's a different way to build a real online storefront, selling products already in demand? A way to earn supplemental income with